# 🟠 Chapter 7: Working with Text Data
**Reference:** *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller & Sarah Guido

---

## 7.1 Text Representation: The Bag-of-Words Model
Machine Learning models cannot parse raw text strings directly. To feed text data into an algorithm, we must first translate it into a structured, numerical format. The most common and foundational method for this is the **Bag-of-Words (BoW)** model.

In a Bag-of-Words representation, we discard the original grammar, sentence structure, and word order of the document. Instead, we simply count how many times each unique word appears across our dataset. The process consists of three main steps:
1. **Tokenization:** Splitting the raw document strings into individual units called tokens (words).
2. **Vocabulary Building:** Collecting all unique words that appear across all documents to form a global index.
3. **Encoding:** Mapping each document into a numerical vector where each entry counts the frequency of a vocabulary word.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Sample toy dataset containing two sentences (documents)
bards_words = [
    "The fool doth think he is wise,",
    "but the wise man knows himself to be a fool"
]

# Instantiate the vectorizer tool
vect = CountVectorizer()
vect.fit(bards_words)

print("Vocabulary size:", len(vect.vocabulary_))
print("Vocabulary content (Word to Index mapping):\n", vect.vocabulary_)

# Transform the raw text into a numerical sparse matrix representation
bag_of_words = vect.transform(bards_words)
print("\nBag-of-Words Matrix Shape:", bag_of_words.shape)
print("Dense Array Representation:\n", bag_of_words.toarray())

print("\nNote: Scikit-learn outputs a SciPy sparse matrix by default to save memory,")
print("as most documents contain only a tiny fraction of the global vocabulary tokens.")

## 7.2 Term Frequency-Inverse Document Frequency (TF-IDF)
While the standard Bag-of-Words model is effective, it has a significant drawback: words that appear very frequently across all documents (such as "the", "is", "and", "a") end up dominant. However, these common tokens carry very little semantic value for classification or clustering.

To suppress this noise, we use **Term Frequency-Inverse Document Frequency (TF-IDF)** scaling. This algorithm scales the frequency of a word based on how unique it is to a specific document:
- **Term Frequency (TF):** Rewards words that appear frequently within a specific document.
- **Inverse Document Frequency (IDF):** Penalizes words that appear frequently across *all* documents in the corpus.

The mathematical product guarantees that high weights are reserved exclusively for words that provide strong, distinctive information about individual documents.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a representative synthetic text corpus
corpus = [
    "machine learning is fascinating and machine learning is the future",
    "the future of artificial intelligence is bright",
    "deep learning algorithms are built on neural networks",
    "neural networks can learn complex patterns from data"
]

# Instantiate the vectorizer, stripping out standard english connector tokens (stopwords)
tfidf_vect = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vect.fit_transform(corpus)

# Extract tokens and sum their cumulative scaled weights across the corpus
features = tfidf_vect.get_feature_names_out()
tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).ravel()

# Sort feature tokens to identify the top 10 highest-ranked terms
sorted_indices = np.argsort(tfidf_scores)
top_features = features[sorted_indices[-10:]]
top_scores = tfidf_scores[sorted_indices[-10:]]

# Visualize TF-IDF performance metrics
plt.figure(figsize=(10, 5))
plt.barh(top_features, top_scores, color='teal', alpha=0.8)
plt.title("Top 10 Terms Ranked by Cumulative TF-IDF Score Across Corpus", fontsize=14)
plt.xlabel("Cumulative TF-IDF Metric Weight", fontsize=12)
plt.ylabel("Vocabulary Tokens", fontsize=12)
plt.tight_layout()
plt.show()

print("Analysis:")
print("Notice how domain-specific context words like 'machine' and 'networks' receive high weights,")
print("while generic terms like 'is' and 'the' were automatically discarded by the pipeline.")

## 7.3 Topic Modeling with Latent Dirichlet Allocation (LDA)
In many real-world applications, we deal with vast collections of unlabelled text documents and want to discover what they are talking about. This unsupervised extraction task is called **Topic Modeling**.

**Latent Dirichlet Allocation (LDA)** is a probabilistic model that works by assuming that each document is composed of a mixture of multiple latent topics, and each topic is characterized by a specific distribution of words. Geometrically, it uncovers these overlapping word clusters to present them as human-interpretable subjects.

### Document-to-Topic Generative Concept
The diagram below visualizes how LDA conceptualizes documents as mixtures of latent topics, which are in turn defined by word distributions:



In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

# LDA processes raw word counts rather than TF-IDF matrices
vect_lda = CountVectorizer(stop_words='english')
X_lda = vect_lda.fit_transform(corpus)

# Configure LDA to isolate 2 distinctive latent topics (n_components=2)
lda = LatentDirichletAllocation(n_components=2, learning_method="batch", random_state=0)
document_topics = lda.fit_transform(X_lda)

# Extract vocabulary tokens and individual topic component profiles
topics = lda.components_
feature_names = vect_lda.get_feature_names_out()

# Generate side-by-side subplots visualizing the top 5 most important words for each topic
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False)
for topic_idx, topic in enumerate(topics):
    top_features_ind = topic.argsort()[:-6:-1]
    top_features = [feature_names[i] for i in top_features_ind]
    weights = topic[top_features_ind]

    ax = axes[topic_idx]
    ax.barh(top_features, weights, height=0.6, color='orange', alpha=0.8)
    ax.set_title(f'Latent Topic Profile {topic_idx + 1}', fontsize=14, weight='bold')
    ax.invert_yaxis()
    ax.tick_params(axis='both', labelsize=12)
    ax.set_xlabel("Internal Importance Weight Metrics")

plt.suptitle("LDA Topic Modeling Feature Extraction Results", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("LDA has automatically clustered our corpus words into two distinct semantic themes:")
print("- Topic 1 focuses heavily on structural infrastructure components ('learning', 'machine', 'future').")
print("- Topic 2 isolates network-oriented technical terms ('networks', 'neural', 'data').")

## 7.4 Summary and Pipeline Considerations
When working with text data, keep the following core concepts from the book in mind:
- **N-Grams:** By default, tokenization looks at single words (unigrams). To capture context like "not good" vs. "good", configure `ngram_range` to include bigrams or trigrams.
- **Stemming and Lemmatization:** Dropping word suffixes (converting "running" or "runs" to "run") reduces vocabulary size and combines duplicate meanings. While Scikit-learn doesn't include these natively, they can be easily integrated via `nltk` or `spacy` preprocessors.
- **Sparsity Optimization:** Text matrices are highly sparse. Always maintain them using SciPy sparse formats to protect RAM from overflow.